In [2]:
from pde import DiffusionPDE, ScalarField, UnitGrid, CartesianGrid, MemoryStorage, PDE, movie, plot_kymograph
import numpy as np
import gc

In [3]:
def generate_string(params, opp='cos', var='x'):
    string = ""
    for idx, (weight, freq) in enumerate(params):
        if(idx == 0):
            string += f"{weight:.3f}*{opp}({freq:.3f}*2*pi*{var})"
        else:
            string += f"+{weight:.3f}*{opp}({freq:.3f}*2*pi*{var})"
    return string

In [4]:
def generate_grid(N:int, x_periodic:bool, y_periodic:bool):
    grid = UnitGrid([N, N], periodic=[x_periodic, y_periodic])
    return grid

In [8]:
cos_x_params = [(0.4, 0.25),(0.3, 0.3)]
sin_x_params = [(0.25, 0.1),(0.0, 0.3),(0.5, 0.1)]
cos_y_params = [(0.1, 0.1),(0.0, 0.1),(0.2, 0.01),(0.3, 0.1)]
sin_y_params = [(0.25, 0.125),]

def generate_sinusoidal_weights(cos_x_params, sin_x_params, cos_y_params, sin_y_params):
    cos_x_inital_weights = [(np.random.normal(mu, sigma), (idx + 1) / N) for idx, (mu, sigma) in enumerate(cos_x_params)]
    sin_x_inital_weights = [(np.random.normal(mu, sigma), (idx + 1) / N) for idx, (mu, sigma) in enumerate(sin_x_params)]
    cos_y_inital_weights = [(np.random.normal(mu, sigma), (idx + 1) / N) for idx, (mu, sigma) in enumerate(cos_y_params)]
    sin_y_inital_weights = [(np.random.normal(mu, sigma), (idx + 1) / N) for idx, (mu, sigma) in enumerate(sin_y_params)]
    return cos_x_inital_weights, sin_x_inital_weights, cos_y_inital_weights, sin_y_inital_weights

def generate_sinusoidal_equation(cos_x_inital_weights, sin_x_inital_weights, cos_y_inital_weights, sin_y_inital_weights):
    init_equation = generate_string(cos_x_inital_weights)
    init_equation += '+' + generate_string(sin_x_inital_weights, opp='sin')
    init_equation += '+' + generate_string(cos_y_inital_weights, var='y')
    init_equation += '+' + generate_string(sin_y_inital_weights, opp='sin', var='y')
    return init_equation

def generate_periodic_init_state(grid, init_equation):
    state = ScalarField.from_expression(grid, init_equation)
    state.data = np.maximum(np.minimum(state.data, 1.0), 0.0)
    return state

def generate_random_init_state(grid):
    state = ScalarField.random_uniform(grid, 0.0, 1.0)
    return state

def generate_state_from_data(grid, data):
    state = ScalarField(grid, data=data)

In [6]:
def get_nu_sample():
    nu = np.random.uniform(low=0.1, high=0.4)
    print(nu)
    return nu

def get_fixed_nu():
    return 0.2

In [7]:

def generate_boundary_conditions(x_periodic:bool, y_periodic:bool):
    if(x_periodic):
        bc_x = 'periodic'
    else:
        bc_x = {"derivative": 0.0}
    if(y_periodic):
        bc_y = 'periodic'
    else:
        bc_y = {"derivative": 0.0}
    return [bc_x, bc_y]

def get_burgers_pde(nu, bcs):
    eq = PDE(
        {'u':f'{nu} * (laplace(u) + laplace(u)) - u * (getx(gradient(u)) + gety(gradient(u)))'},
        user_funcs={'getx':lambda arr: arr[0], 'gety':lambda arr: arr[1]},
        bc=bcs
    )
    return eq

In [ ]:
def solve_pde(eq, state):
    example = []
    storage = MemoryStorage()
    results = eq.solve(state, t_range=10, dt=0.25, tracker=["progress", storage.tracker(0.25)])
    for time_step, field in storage.items():
        example.append(np.expand_dims(field.data, axis=0))
    example = np.expand_dims(np.concatenate(example.copy()), axis=0)
    # movie(storage, filename="example_burgers.mov")
    print(example.min(), example.max(), example.mean(), example.std())
    if(np.any(np.isnan(example))):
        raise Exception('')
    return example

In [ ]:
# get the starting grid
# get the intial conditions
# solve the PDE periodically for N steps
# solve the PDE non periodically for N steps

# for each example we will have Nl large steps
# then for each large step we will have Ns small steps
# this should work
# is this even cool tho?


In [ ]:
def generate_example(N, x_periodic, y_periodic, random_nu_sample:bool=True, periodic_init:bool=True):
    # get the grid
    grid = generate_grid(N, x_periodic, y_periodic)
    # get the initial state
    if(periodic_init):
        weights = generate_sinusoidal_weights(cos_x_params, sin_x_params, cos_y_params, sin_y_params)
        init_equation = generate_sinusoidal_equation(*weights)
        state = generate_periodic_init_state(grid, init_equation)
    else:
        init_equation = 'uniform 0.1 to 0.4'
        state = generate_random_init_state(grid)
    # get the hyperparams
    if(random_nu_sample):
        nu = get_nu_sample()
    else:
        nu = get_fixed_nu()
    # get boundary conditions
    boundary_conditions = generate_boundary_conditions(x_periodic, y_periodic)
    # get the pde
    eq = get_burgers_pde(nu, boundary_conditions)
    # solve the pde
    solution = solve_pde(eq, state)
    # delete the things we don't need
    del grid
    del state
    del eq
    gc.collect()
    return solution, [nu, init_equation]

N = 128
X_PERIOD = False
Y_PERIOD = False
RANDOM_NU = True 
PERIODIC_INIT = True

def generate_all_examples(n=50):
    data, meta = [], []
    for idx in range(n):
        solution, info = generate_example(N, X_PERIOD, Y_PERIOD, random_nu_sample=RANDOM_NU, periodic_init=PERIODIC_INIT)
        data.append(solution)
        meta.append(info)
    return np.concatenate(data), np.concatenate(meta)

train_data, train_meta = generate_all_examples(n=50)
val_data, val_meta = generate_all_examples(n=8)
test_data, test_meta = generate_all_examples(n=8)

name = f'burgers_{N}'
if(PERIODIC_INIT):
    name += '_period_init'
else:
    name += '_non_period_init'

if(X_PERIOD):
    name += '_x_period'
else:
    name += '_x_non_period'

if(Y_PERIOD):
    name += '_y_period'
else:
    name += '_y_non_period'

if(RANDOM_NU):
    name += '_varying_nu'
else:
    name += "_fixed_nu"

name += '.npy'
print(name)

with open(name, mode='wb') as f:
    np.save(f, train_data, allow_pickle=True)
    np.save(f, val_data, allow_pickle=True)
    np.save(f, test_data, allow_pickle=True)

    np.save(f, train_meta, allow_pickle=True)
    np.save(f, val_meta, allow_pickle=True)
    np.save(f, test_meta, allow_pickle=True)
    
train_data.shape, val_data.shape, test_data.shape